# Silver transform stream

Purpose: read the append-only Bronze GE validated stream, transform and deduplicate it, and land a clean Silver streaming table plus batch metrics.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"

STORAGE_ACCOUNT = "streanmingdatasta"
LAKEHOUSE_CONTAINER = "lakehouse"
BASE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/external/hant-catalog"

BRONZE_VALIDATED_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.bronze_vehicle_positions_validated_stream"
BRONZE_VALIDATED_PATH_DEFAULT = f"{BASE_PATH}/bronze/bronze_validated_stream/hsl_vehicle_positions"

SILVER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_vehicle_positions_cleaned"
SILVER_METRICS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_vehicle_positions_cleaned_metrics"
SILVER_OPS_EVENTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_operational_events"

SILVER_PATH = f"{BASE_PATH}/silver/silver_hsl_vehicle_positions_cleaned"
SILVER_METRICS_PATH = f"{BASE_PATH}/silver/silver_metrics/hsl_vehicle_positions_cleaned"
SILVER_OPS_EVENTS_PATH = f"{BASE_PATH}/silver/silver_metrics/operational_events"

CHECKPOINT_BASE = f"{BASE_PATH}/silver/checkpoints"
CHECKPOINT_CLEAN_PATH = f"{CHECKPOINT_BASE}/silver_hsl_vehicle_positions_cleaned_ge_stream"
CHECKPOINT_METRICS_PATH = f"{CHECKPOINT_BASE}/silver_hsl_vehicle_positions_cleaned_metrics_ge_stream"

TRIGGER_INTERVAL = "10 seconds"
WATERMARK_DELAY = "10 minutes"


In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {BRONZE_VALIDATED_TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {SILVER_METRICS_TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {SILVER_OPS_EVENTS_TABLE}")

# dbutils.fs.rm(CHECKPOINT_CLEAN_PATH, True)
# dbutils.fs.rm(CHECKPOINT_METRICS_PATH, True)

In [0]:
def get_task_value(task_key: str, key: str, debug_value=None):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key, debugValue=debug_value)
    except Exception:
        return debug_value


bronze_gate_status = get_task_value("bronze_validate_ge", "bronze_gate_status", "STREAMING")
bronze_gate_reason = get_task_value("bronze_validate_ge", "bronze_gate_reason", "streaming_bronze_ge_not_yet_reported")
bronze_validated_path = get_task_value("bronze_validate_ge", "bronze_validated_path", BRONZE_VALIDATED_PATH_DEFAULT)
bronze_validation_run_id = get_task_value("bronze_validate_ge", "bronze_validation_run_id", "streaming_run_pending")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {SILVER_OPS_EVENTS_TABLE} (
    event_ts TIMESTAMP,
    layer STRING,
    upstream_layer STRING,
    upstream_run_id STRING,
    status STRING,
    message STRING,
    source_path STRING
)
USING DELTA
LOCATION "{SILVER_OPS_EVENTS_PATH}"
''')

(
    spark.createDataFrame([{
        "event_ts": None,
        "layer": "silver_transform_stream",
        "upstream_layer": "bronze_validate_ge_stream",
        "upstream_run_id": bronze_validation_run_id,
        "status": bronze_gate_status,
        "message": bronze_gate_reason,
        "source_path": bronze_validated_path,
    }], schema='''
        event_ts timestamp,
        layer string,
        upstream_layer string,
        upstream_run_id string,
        status string,
        message string,
        source_path string
    ''')
    .withColumn("event_ts", F.current_timestamp())
    .write.format("delta")
    .mode("append")
    .save(SILVER_OPS_EVENTS_PATH)
)


In [0]:
def normalize_str(col_expr):
    return (
        F.when(col_expr.isNull(), None)
        .otherwise(F.regexp_replace(F.trim(col_expr.cast("string")), r"^0+(?=\d)", ""))
    )


envelope_schema = T.StructType([
    T.StructField("ingest_ts_utc", T.StringType(), True),
    T.StructField("source", T.StringType(), True),
    T.StructField("mqtt", T.StructType([
        T.StructField("host", T.StringType(), True),
        T.StructField("port", T.IntegerType(), True),
        T.StructField("topic", T.StringType(), True),
        T.StructField("qos", T.IntegerType(), True),
        T.StructField("retain", T.BooleanType(), True),
    ]), True),
    T.StructField("topic_parsed", T.StructType([
        T.StructField("raw_topic", T.StringType(), True),
        T.StructField("prefix", T.StringType(), True),
        T.StructField("version", T.StringType(), True),
        T.StructField("journey_type", T.StringType(), True),
        T.StructField("temporal_type", T.StringType(), True),
        T.StructField("event_type", T.StringType(), True),
        T.StructField("transport_mode", T.StringType(), True),
        T.StructField("operator_id", T.StringType(), True),
        T.StructField("vehicle_number", T.StringType(), True),
        T.StructField("route_id", T.StringType(), True),
        T.StructField("direction_id", T.StringType(), True),
        T.StructField("headsign", T.StringType(), True),
        T.StructField("start_time", T.StringType(), True),
        T.StructField("next_stop_id", T.StringType(), True),
        T.StructField("geo_tail", T.StringType(), True),
    ]), True),
    T.StructField("payload", T.StructType([
        T.StructField("VP", T.StructType([
            T.StructField("desi", T.StringType(), True),
            T.StructField("dir", T.StringType(), True),
            T.StructField("oper", T.IntegerType(), True),
            T.StructField("veh", T.IntegerType(), True),
            T.StructField("tst", T.StringType(), True),
            T.StructField("tsi", T.LongType(), True),
            T.StructField("spd", T.DoubleType(), True),
            T.StructField("hdg", T.IntegerType(), True),
            T.StructField("lat", T.DoubleType(), True),
            T.StructField("long", T.DoubleType(), True),
            T.StructField("acc", T.DoubleType(), True),
            T.StructField("dl", T.IntegerType(), True),
            T.StructField("odo", T.IntegerType(), True),
            T.StructField("drst", T.IntegerType(), True),
            T.StructField("oday", T.StringType(), True),
            T.StructField("jrn", T.IntegerType(), True),
            T.StructField("line", T.IntegerType(), True),
            T.StructField("start", T.StringType(), True),
            T.StructField("loc", T.StringType(), True),
            T.StructField("stop", T.IntegerType(), True),
            T.StructField("route", T.StringType(), True),
            T.StructField("occu", T.IntegerType(), True),
        ]), True),
    ]), True),
])

try:
    bronze_validated_stream_df = spark.readStream.table(BRONZE_VALIDATED_TABLE)
except Exception:
    bronze_validated_stream_df = spark.readStream.format("delta").load(bronze_validated_path)

parsed_df = (
    bronze_validated_stream_df
    .withColumn("envelope", F.from_json(F.col("raw_json"), envelope_schema))
    .withColumn("vp", F.col("envelope.payload.VP"))
)


In [0]:
silver_clean_df = (
    parsed_df
    .select(
        "topic", "partition", "offset", "eventhub_enqueued_ts", "message_key", "raw_json",
        "bronze_ingest_ts", "ingest_date", "parse_ok", "parse_error", "source",
        "producer_ingest_ts_utc", "mqtt_topic", "mqtt_qos", "mqtt_retain",
        "validation_run_id", "validation_date", "validation_status",
        F.col("envelope.mqtt.host").alias("mqtt_host"),
        F.col("envelope.mqtt.port").alias("mqtt_port"),
        F.col("envelope.topic_parsed.event_type").alias("topic_event_type"),
        F.col("envelope.topic_parsed.transport_mode").alias("topic_transport_mode"),
        F.col("envelope.topic_parsed.operator_id").alias("topic_operator_id"),
        F.col("envelope.topic_parsed.vehicle_number").alias("topic_vehicle_number"),
        F.col("envelope.topic_parsed.route_id").alias("topic_route_id"),
        F.col("envelope.topic_parsed.direction_id").alias("topic_direction_id"),
        F.col("envelope.topic_parsed.headsign").alias("topic_headsign"),
        F.col("envelope.topic_parsed.start_time").alias("topic_start_time"),
        F.col("envelope.topic_parsed.next_stop_id").alias("topic_next_stop_id"),
        F.col("vp.desi").alias("desi"),
        F.col("vp.dir").alias("dir"),
        F.col("vp.oper").cast("string").alias("operator_id"),
        F.col("vp.veh").cast("string").alias("vehicle_number"),
        F.col("vp.tst").alias("event_ts_raw"),
        F.to_timestamp("vp.tst").alias("event_ts"),
        F.col("vp.tsi").cast("long").alias("event_ts_unix"),
        F.col("vp.spd").cast("double").alias("speed"),
        F.col("vp.hdg").cast("int").alias("heading"),
        F.col("vp.lat").cast("double").alias("latitude"),
        F.col("vp.long").cast("double").alias("longitude"),
        F.col("vp.acc").cast("double").alias("acceleration"),
        F.col("vp.dl").cast("int").alias("delay_sec"),
        F.col("vp.odo").cast("int").alias("odometer_m"),
        F.col("vp.drst").cast("int").alias("door_status"),
        F.to_date("vp.oday").alias("operating_day"),
        F.col("vp.jrn").cast("int").alias("journey_id"),
        F.col("vp.line").cast("string").alias("line_id"),
        F.col("vp.start").alias("journey_start_time"),
        F.col("vp.loc").alias("location_source"),
        F.col("vp.stop").cast("string").alias("stop_id"),
        F.col("vp.route").alias("payload_route_id"),
        F.col("vp.occu").cast("int").alias("occupancy"),
    )
    .withColumn("topic_vehicle_number_norm", normalize_str(F.col("topic_vehicle_number")))
    .withColumn("topic_operator_id_norm", normalize_str(F.col("topic_operator_id")))
    .withColumn("route_id", F.coalesce(F.col("payload_route_id"), F.col("topic_route_id"), F.col("desi")))
    .withColumn("direction_id", F.coalesce(F.col("dir"), F.col("topic_direction_id")))
    .withColumn("vehicle_id", F.concat_ws("_", F.coalesce(F.col("operator_id"), F.col("topic_operator_id_norm")), F.coalesce(F.col("vehicle_number"), F.col("topic_vehicle_number_norm"))))
    .withColumn("business_key", F.concat_ws("|", F.coalesce(F.col("vehicle_id"), F.lit("")), F.coalesce(F.col("event_ts_unix").cast("string"), F.lit("")), F.coalesce(F.col("route_id"), F.lit("")), F.coalesce(F.col("direction_id"), F.lit(""))))
    .withColumn("silver_ingest_ts", F.current_timestamp())
    .withColumn("dedup_event_ts", F.coalesce(F.col("event_ts"), F.col("eventhub_enqueued_ts"), F.col("bronze_ingest_ts"), F.current_timestamp()))
    .withColumn("silver_event_date", F.to_date(F.coalesce(F.col("event_ts"), F.col("eventhub_enqueued_ts"), F.col("bronze_ingest_ts"))))
    .withColumn("event_to_enqueue_delay_sec", (F.col("eventhub_enqueued_ts").cast("long") - F.col("event_ts").cast("long")).cast("long"))
    .withColumn("enqueue_to_bronze_delay_sec", (F.col("bronze_ingest_ts").cast("long") - F.col("eventhub_enqueued_ts").cast("long")).cast("long"))
    .withColumn("bronze_to_silver_delay_sec", (F.col("silver_ingest_ts").cast("long") - F.col("bronze_ingest_ts").cast("long")).cast("long"))
    .withColumn("event_to_silver_delay_sec", (F.col("silver_ingest_ts").cast("long") - F.col("event_ts").cast("long")).cast("long"))
)

silver_dedup_df = silver_clean_df.withWatermark("dedup_event_ts", WATERMARK_DELAY).dropDuplicates(["business_key"])


In [0]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
    topic STRING, partition INT, offset BIGINT, eventhub_enqueued_ts TIMESTAMP, message_key STRING, raw_json STRING,
    bronze_ingest_ts TIMESTAMP, ingest_date DATE, parse_ok BOOLEAN, parse_error STRING, source STRING, producer_ingest_ts_utc STRING,
    mqtt_topic STRING, mqtt_qos INT, mqtt_retain BOOLEAN, validation_run_id STRING, validation_date STRING, validation_status STRING,
    mqtt_host STRING, mqtt_port INT, topic_event_type STRING, topic_transport_mode STRING, event_type STRING, transport_mode STRING,
    topic_operator_id STRING, topic_vehicle_number STRING, topic_vehicle_number_norm STRING, topic_operator_id_norm STRING, topic_route_id STRING,
    topic_direction_id STRING, topic_headsign STRING, topic_start_time STRING, topic_next_stop_id STRING, desi STRING, dir STRING,
    operator_id STRING, vehicle_number STRING, event_ts_raw STRING, event_ts TIMESTAMP, event_ts_unix BIGINT, speed DOUBLE, heading INT,
    latitude DOUBLE, longitude DOUBLE, acceleration DOUBLE, delay_sec INT, odometer_m INT, door_status INT, operating_day DATE, journey_id INT,
    line_id STRING, journey_start_time STRING, location_source STRING, stop_id STRING, payload_route_id STRING, occupancy INT, route_id STRING,
    direction_id STRING, vehicle_id STRING, business_key STRING, silver_ingest_ts TIMESTAMP, dedup_event_ts TIMESTAMP, silver_event_date DATE,
    event_to_enqueue_delay_sec BIGINT, enqueue_to_bronze_delay_sec BIGINT, bronze_to_silver_delay_sec BIGINT, event_to_silver_delay_sec BIGINT
)
USING DELTA
PARTITIONED BY (silver_event_date)
LOCATION "{SILVER_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {SILVER_METRICS_TABLE} (
    batch_id BIGINT, batch_ts TIMESTAMP, total_rows BIGINT, event_ts_null_rows BIGINT, latitude_null_rows BIGINT, longitude_null_rows BIGINT,
    duplicate_business_key_rows BIGINT, topic_payload_route_mismatch_rows BIGINT, topic_payload_vehicle_mismatch_rows BIGINT,
    stale_event_rows BIGINT, future_event_rows BIGINT, bronze_gate_status STRING, bronze_validation_run_id STRING
)
USING DELTA
LOCATION "{SILVER_METRICS_PATH}"
''')

DataFrame[]

In [0]:
def write_silver_metrics(batch_df, batch_id: int):
    if batch_df.isEmpty():
        return

    duplicate_business_keys_df = batch_df.groupBy("business_key").count().filter(F.col("count") > 1)
    duplicate_row_count = duplicate_business_keys_df.agg(F.sum(F.col("count") - 1).alias("duplicate_rows")).collect()[0]["duplicate_rows"]
    upstream_validation_run_id = batch_df.agg(F.max("validation_run_id").alias("validation_run_id")).collect()[0]["validation_run_id"]

    (
        batch_df.agg(
            F.count("*").alias("total_rows"),
            F.sum(F.when(F.col("event_ts").isNull(), 1).otherwise(0)).cast("bigint").alias("event_ts_null_rows"),
            F.sum(F.when(F.col("latitude").isNull(), 1).otherwise(0)).cast("bigint").alias("latitude_null_rows"),
            F.sum(F.when(F.col("longitude").isNull(), 1).otherwise(0)).cast("bigint").alias("longitude_null_rows"),
            F.sum(F.when(F.col("topic_route_id").isNotNull() & F.col("payload_route_id").isNotNull() & (F.col("topic_route_id") != F.col("payload_route_id")), 1).otherwise(0)).cast("bigint").alias("topic_payload_route_mismatch_rows"),
            F.sum(F.when(F.col("topic_vehicle_number_norm").isNotNull() & F.col("vehicle_number").isNotNull() & (F.col("topic_vehicle_number_norm") != F.col("vehicle_number")), 1).otherwise(0)).cast("bigint").alias("topic_payload_vehicle_mismatch_rows"),
            F.sum(F.when(F.col("event_to_silver_delay_sec") > F.lit(300), 1).otherwise(0)).cast("bigint").alias("stale_event_rows"),
            F.sum(F.when(F.col("event_to_silver_delay_sec") < F.lit(-120), 1).otherwise(0)).cast("bigint").alias("future_event_rows"),
        )
        .withColumn("duplicate_business_key_rows", F.lit(int(duplicate_row_count or 0)).cast("bigint"))
        .withColumn("batch_id", F.lit(int(batch_id)).cast("bigint"))
        .withColumn("batch_ts", F.current_timestamp())
        .withColumn("bronze_gate_status", F.lit("STREAMING_GE"))
        .withColumn("bronze_validation_run_id", F.lit(upstream_validation_run_id))
        .write.format("delta").mode("append").save(SILVER_METRICS_PATH)
    )


In [0]:
for q in spark.streams.active:
    if q.name in {"silver_hsl_vehicle_positions_cleaned", "silver_hsl_vehicle_positions_cleaned_metrics"}:
        q.stop()

silver_clean_query = (
    silver_dedup_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_CLEAN_PATH)
    .queryName("silver_hsl_vehicle_positions_cleaned")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable(SILVER_TABLE)
)

silver_metrics_query = (
    silver_clean_df.writeStream
    .foreachBatch(write_silver_metrics)
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_METRICS_PATH)
    .queryName("silver_hsl_vehicle_positions_cleaned_metrics")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)


In [0]:
for q in spark.streams.active:
    if q.name in {"silver_hsl_vehicle_positions_cleaned", "silver_hsl_vehicle_positions_cleaned_metrics"}:
        print("NAME:", q.name)
        print("ID:", q.id)
        print("IS ACTIVE:", q.isActive)
        print("STATUS:", q.status)
        print("LAST PROGRESS:", q.lastProgress)
        print("EXCEPTION:", q.exception())
        print("-" * 80)

silver_clean_query.awaitTermination()
silver_metrics_query.awaitTermination()


NAME: silver_hsl_vehicle_positions_cleaned
ID: 8f4f09e5-f61e-4a2c-9a66-758bef4cf4fd
IS ACTIVE: True
STATUS: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
LAST PROGRESS: None
EXCEPTION: None
--------------------------------------------------------------------------------
NAME: silver_hsl_vehicle_positions_cleaned_metrics
ID: b8375889-9e26-489f-a2b8-1009d292f84a
IS ACTIVE: True
STATUS: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
LAST PROGRESS: None
EXCEPTION: None
--------------------------------------------------------------------------------


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can